In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv('Telco Customer Churn.csv')
df.head()


EDA

In [ ]:
df.columns.values

In [ ]:
print(df.info())
print(df.shape)
print(df.describe())
print(df.dtypes)
print(df.isnull().sum())

In [ ]:
df.Churn.value_counts()
(df["Churn"]=="Yes").mean()*100

In [ ]:
# Convert dtype of TotalCharges to numeric
print(df["TotalCharges"].dtype)


df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

print("\nAfter Conversion:")
print(df["TotalCharges"].dtype)


In [ ]:
#filling the blank values in TotalCharges with median value of the column
df["TotalCharges"].fillna(
    df["TotalCharges"].median(),
    inplace=True
)

print("\nMISSING VALUES AFTER FIX")
print(df.isnull().sum())

In [ ]:
#churn distribution
print("\nCHURN DISTRIBUTION")

print(df["Churn"].value_counts())

# Plot

plt.figure(figsize=(6,4))

sns.countplot(
    x="Churn",
    data=df
)

plt.title("Churn Distribution")
plt.show()


In [ ]:
# Calculate churn rate
churn_rate = (
    (df["Churn"]=="Yes").mean()*100
)
print(f"\nChurn Rate: {churn_rate:.2f}%")

In [ ]:
numerical_cols = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

for col in numerical_cols:
    plt.figure(figsize=(8,4))
    sns.histplot(
        data=df,
        x=col,
        hue="Churn",
        kde=True
    )

    plt.title(f"{col} vs Churn")
    plt.show()

In [ ]:
#CHURN RATE PER CATEGORICAL FEATURE
#churn rate by payment method
pm = df.groupby("PaymentMethod")["Churn"]\
       .apply(lambda x: (x=="Yes").mean()*100)

print(pm.round(1))

plt.figure(figsize=(8,4))

pm.plot(kind="bar")

plt.title("Payment Method vs Churn")
plt.ylabel("Churn Percentage")

plt.show()

In [ ]:
#internet service vs churn
internet = df.groupby("InternetService")["Churn"]\
             .apply(lambda x: (x=="Yes").mean()*100)

print(internet.round(1))

plt.figure(figsize=(6,4))

internet.plot(kind="bar")

plt.title("Internet Service vs Churn")
plt.ylabel("Churn Percentage")

plt.show()

In [ ]:
#tech support vs churn
tech = df.groupby("TechSupport")["Churn"]\
         .apply(lambda x: (x=="Yes").mean()*100)

print(tech.round(1))

plt.figure(figsize=(6,4))

tech.plot(kind="bar")

plt.title("Tech Support vs Churn")
plt.ylabel("Churn Percentage")

plt.show()

In [ ]:
#correlation heatmap for numerical columns and churn 
# Convert Churn into numeric

df["Churn_num"] = df["Churn"].map({
    "No":0,
    "Yes":1
})

# Select numerical columns

corr_cols = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "SeniorCitizen",
    "Churn_num"
]

# Correlation matrix

corr = df[corr_cols].corr()
print(corr)

plt.figure(figsize=(8,5))

sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="RdYlGn",
    center=0
)

plt.title("Correlation Heatmap")
plt.show()

PREPROCESSING AND FEATURE ENGINEERING

In [ ]:
# replace special values in the following columns with "No" to make it easier for analysis
replace_columns = [
    "MultipleLines",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

for col in replace_columns:

    df[col] = df[col].replace(
        "No phone service",
        "No"
    )

    df[col] = df[col].replace(
        "No internet service",
        "No"
    )

print("\nSpecial values replaced successfully")

In [ ]:
# drop the customerID column as it is not useful for prediction
df = df.drop("customerID", axis=1 , inplace=True)

In [ ]:
# Encode churn column to numeric values 
df['Churn'] = df['Churn'].map({'Yes':1,'No':0})
print(df['Churn'].value_counts())

In [ ]:
# Separate features and target variable
X = df.drop(columns=['Churn'])
y = df['Churn']

print("X Shape:", X.shape)
print("y Shape:", y.shape)


In [ ]:
#Column separation into numerical and categorical columns
num_cols = X.select_dtypes(
    include=['int64','float64']).columns.tolist()

cat_cols = X.select_dtypes(
    include=['object']).columns.tolist()

print(f"Numerical: {len(num_cols)} features")
print(f"Categorical: {len(cat_cols)} features")


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    StandardScaler, OneHotEncoder)
from sklearn.compose import ColumnTransformer
preprocessor = ColumnTransformer([
  ('num', StandardScaler(), num_cols),
  ('cat', OneHotEncoder(
      handle_unknown='ignore',
      sparse_output=False), cat_cols)
])


In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2,
    stratify=y,   # ← CRITICAL for imbalanced data
    random_state=42)
print(f"Train: {X_train.shape}")  # (5634, 19)
print(f"Test:  {X_test.shape}")   # (1409, 19)

print("\nTrain Churn %")
print(y_train.mean()*100)

print("\nTest Churn %")
print(y_test.mean()*100)


In [ ]:
df['tenure_group'] = pd.cut(df['tenure'],
    bins=[0,12,24,48,72],
    labels=['New','Young','Mature','Loyal'])
df['monthly_per_tenure'] = (
    df['MonthlyCharges'] / (df['tenure']+1))
service_cols = ['PhoneService','InternetService',
  'OnlineSecurity','OnlineBackup','TechSupport']
df['num_services'] = (df[service_cols]=='Yes').sum(axis=1)


In [ ]:
import joblib
df.to_csv(
    "../data/processed_telco_churn.csv",
    index=False
)

print("Processed dataset saved")

In [ ]:
joblib.dump(
    preprocessor,
    "../models/preprocessor.pkl"
)

print("Preprocessor saved successfully")

MODEL BUILDING AND TRAINING

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    StratifiedKFold,
     cross_val_score
)

print("All imports successful")

In [ ]:
scale = (
    (y_train == 0).sum() /
    
    (y_train == 1).sum()
)
print("Scale Weight:", scale)

models = {

    "Logistic Regression": LogisticRegression(
    class_weight="balanced",
    max_iter=1000
    ),

    
    "Decision Tree": DecisionTreeClassifier(
    max_depth=6,
    class_weight="balanced",
    random_state=42
    ),

    
    "Random Forest": RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    class_weight="balanced",
    random_state=42
    ),

    
    "Gradient Boosting": GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    random_state=42
    ),

    
    "XGBoost": XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    scale_pos_weight=scale,
    eval_metric="logloss",
    random_state=42
    )
}
print("Models dictionary created")

In [ ]:
# Sklearn Pipeline for each model
pipelines = {}
for name, model in models.items():

    pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
    ])
    pipelines[name] = pipe


print("Pipelines created successfully")
print("\nAvailable Pipelines:")
for key in pipelines.keys():
    print(key)

In [ ]:
results = []

skf = StratifiedKFold(
n_splits=5,
shuffle=True,
random_state=42
)

for name, pipe in pipelines.items():
    print(f"\nRunning CV for {name}...")
    
    scores = cross_val_score(
    pipe,
    X_train,
    y_train,    
    cv=skf,
    scoring="roc_auc"
    )
    
    mean_score = scores.mean()

    print("Scores:", scores)
    print("Mean ROC AUC:", mean_score)

    results.append({
    "Model": name,
    "ROC_AUC": mean_score
    })
results_df = pd.DataFrame(results)

print("\nCROSS VALIDATION RESULTS")

print(results_df.sort_values(
by="ROC_AUC",
ascending=False
))

In [ ]:
model_results = []
best_model = None
best_auc = 0
for name, pipe in pipelines.items():

    print(f"\nTraining {name}...")
    
    pipe.fit(
        
        X_train,
        
        y_train
    )

    y_pred = pipe.predict(X_test)

    y_prob = pipe.predict_proba(X_test)[:,1]
    
    accuracy = accuracy_score(
        
        y_test,
        
        y_pred
    )

    precision = precision_score(
        
        y_test,
        
        y_pred
    )

    recall = recall_score(
        
        y_test,
        
        y_pred
    )

    f1 = f1_score(
        
        y_test,
        y_pred
    )

    auc = roc_auc_score(
        
        y_test,
        
        y_prob
    )
    model_results.append({

        "Model": name,

        "Accuracy": accuracy,

        "Precision": precision,

        "Recall": recall,

        "F1 Score": f1,

        "ROC AUC": auc
    })
    
    if auc > best_auc:
        best_auc = auc
        best_model = pipe
        best_model_name = name
print(f"{name} Completed")

In [ ]:
# Collect metrics and compare models

metrics_df = pd.DataFrame(model_results)
metrics_df = metrics_df.sort_values(
    by="ROC AUC",
    ascending=False
)
print(metrics_df)

In [ ]:
# Classification report for the best model

print("="*50)
print("BEST MODEL")
print("="*50)
print(best_model_name)
print("\nBEST ROC AUC:")
print(best_auc)

y_pred_best = best_model.predict(X_test)

y_prob_best = best_model.predict_proba(X_test)[:,1]

print("\nCLASSIFICATION REPORT")

print(classification_report(
    y_test,
    y_pred_best
))

cm = confusion_matrix(
    y_test,
    y_pred_best
)
print("\nCONFUSION MATRIX") 
print(cm)

Evaluation & Model Selection 

In [ ]:
#ROC Curve for all models

from sklearn.metrics import RocCurveDisplay
import matplotlib.pyplot as plt

# Store trained models

trained_models = {
    "Logistic Regression": pipelines["Logistic Regression"],
    "Decision Tree": pipelines["Decision Tree"],
    "Random Forest": pipelines["Random Forest"],
    "Gradient Boosting": pipelines["Gradient Boosting"],
    "XGBoost": pipelines["XGBoost"]
}

# Fit models if not already fitted

for model in trained_models.values():
    model.fit(X_train, y_train)
    plt.figure(figsize=(8,6))
ax = plt.gca()

for name, model in trained_models.items():
    y_prob = model.predict_proba(X_test)[:,1]
    RocCurveDisplay.from_predictions(
        y_test,
        y_prob,
        ax=ax,
        name=name
    )
plt.plot([0,1],[0,1],'k--',label="Random Guess")
plt.title("ROC Curves - All Models")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

In [ ]:
#Confusion Matrix for the top 2 models

from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

top_models = {
    "Random Forest": pipelines["Random Forest"],
    "XGBoost": pipelines["XGBoost"]
}
for name, model in top_models.items():

    y_pred = model.predict(X_test)
    cm = confusion_matrix(
        y_test,
        y_pred
    )

    plt.figure(figsize=(5,4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues"
    )
    plt.title(f"{name} Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

In [ ]:
#Understand the confusion matrix values
from sklearn.metrics import confusion_matrix

# Best model
best_model = pipelines["XGBoost"]
y_pred = best_model.predict(X_test)
cm = confusion_matrix(
    y_test,
    y_pred
)
TN, FP, FN, TP = cm.ravel()

print("="*40)
print("CONFUSION MATRIX VALUES")
print("="*40)
print(f"True Negative : {TN}")
print(f"False Positive: {FP}")
print(f"False Negative: {FN}")
print(f"True Positive : {TP}")

In [ ]:
#Performance metrics for the best model

from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score

accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred
)

recall = recall_score(
    y_test,
    y_pred
)

f1 = f1_score(
    y_test,
    y_pred
)

auc = roc_auc_score(
    y_test,
    best_model.predict_proba(X_test)[:,1]
)

print("\nAccuracy :", round(accuracy,4))
print("Precision:", round(precision,4))
print("Recall   :", round(recall,4))
print("F1 Score :", round(f1,4))
print("ROC AUC  :", round(auc,4))

In [ ]:
# Tune best model using RandomizedSearchCV

from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline

# Build pipeline

xgb_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("clf", XGBClassifier(
        random_state=42,
        eval_metric="logloss"
    ))
])


param_grid = {
    "clf__n_estimators":[100,200,300,500],
    "clf__max_depth":[3,4,5,6,7],
    "clf__learning_rate":[0.01,0.05,0.1],
    "clf__subsample":[0.7,0.8,0.9,1.0],
    "clf__colsample_bytree":[0.7,0.8,0.9,1.0]
}

search = RandomizedSearchCV(
    estimator=xgb_pipe,
    param_distributions=param_grid,
    n_iter=20,
    scoring="roc_auc",
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1

)

search.fit(X_train, y_train)

print("Best Parameters")
print(search.best_params_)

print("\nBest ROC AUC")
print(search.best_score_)

In [ ]:
#Genrate shap for global feature importance
import shap

# Get trained classifier
best_model = search.best_estimator_

# Transform test data
X_test_processed = best_model.named_steps[
    "preprocessor"
].transform(X_test)

# Get feature names
feature_names = best_model.named_steps[
    "preprocessor"
].get_feature_names_out()

# SHAP Explainer
explainer = shap.TreeExplainer(
    best_model.named_steps["clf"]

)
shap_values = explainer.shap_values(
    X_test_processed
)

# Summary Plot
shap.summary_plot(
    shap_values,
    X_test_processed,
    feature_names=feature_names,
    max_display=15
)

In [ ]:
#Shap waterfall plot for a single sample
sample = 0
shap.plots.waterfall(

    shap.Explanation(
        values=shap_values[sample],
        base_values=explainer.expected_value,
        data=X_test_processed[sample],
        feature_names=feature_names

    )
)

In [ ]:
# Save the best model to a file using joblib and verify its load
import joblib
joblib.dump(

    search.best_estimator_,

    "../models/churn_model_v1.pkl"

)

print("Model Saved Successfully")